# TechJobMCP v2: Train Laya System 1 on Free Cloud GPU

This notebook fine-tunes the **Laya System 1 decision engine** (`convaiinnovations/laya-multilingual`, 322M parameters) using your prepared dataset.

### Recommended Setup:
1. Go to **Runtime** > **Change runtime type**
2. Select **T4 GPU** (free tier on Google Colab)
3. Run the cells sequentially below. Total training time on T4: **~4 to 6 minutes**.

## 1. Install Dependencies & Check GPU

In [ ]:
!pip install -q "transformers>=4.48.0" "datasets>=3.0" safetensors accelerate torch
!nvidia-smi

## 2. Upload Training & Validation JSONL Data
Run this cell to upload:
- `laya_train.jsonl` (from your local `data/training/laya_train.jsonl`)
- `laya_val.jsonl` (from your local `data/holdout/laya_val.jsonl`)

In [ ]:
import os
os.makedirs("data/training", exist_ok=True)
os.makedirs("data/holdout", exist_ok=True)
os.makedirs("data/models", exist_ok=True)

try:
    from google.colab import files
    print("Please upload laya_train.jsonl and laya_val.jsonl:")
    uploaded = files.upload()
    for fname in uploaded:
        if "train" in fname:
            os.replace(fname, "data/training/laya_train.jsonl")
            print(f"Moved {fname} -> data/training/laya_train.jsonl")
        elif "val" in fname or "holdout" in fname:
            os.replace(fname, "data/holdout/laya_val.jsonl")
            print(f"Moved {fname} -> data/holdout/laya_val.jsonl")
except ImportError:
    print("Not running in Google Colab. Ensure laya_train.jsonl and laya_val.jsonl are in data/training and data/holdout.")


## 3. Fine-Tune Laya with On-The-Fly Tokenization & Step-Wise Validation

- Uses **AMP (fp16)** for fast Tensor Core execution.
- Evaluates validation loss and accuracy every 50 steps.
- Automatically preserves the best checkpoint (`best_checkpoint/`).
- Runs **Grid-Search Temperature Calibration** ($T \in [0.1, 5.0]$) to minimize ECE (Expected Calibration Error).

In [ ]:
import json, os, tarfile
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load datasets
train_records = [json.loads(line) for line in open("data/training/laya_train.jsonl")]
val_records = [json.loads(line) for line in open("data/holdout/laya_val.jsonl")]
print(f"Loaded {len(train_records)} train records, {len(val_records)} validation records.")

BASE_MODEL = "convaiinnovations/laya-multilingual"
try:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

class LayaDataset(Dataset):
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        item = self.records[idx]
        opts = " | ".join(f"[{i}] {opt}" for i, opt in enumerate(item["options"]))
        prompt = f"Context:\n{item['state']}\n\nInstruction: {item['instruction']}\nOptions:\n{opts}"
        return {"text": prompt, "label": int(item["label"])}

def collate_fn(batch):
    texts = [b["text"] for b in batch]
    labels = [b["label"] for b in batch]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    enc["labels"] = torch.tensor(labels, dtype=torch.long)
    return enc

train_loader = DataLoader(LayaDataset(train_records), batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(LayaDataset(val_records), batch_size=8, shuffle=False, collate_fn=collate_fn)

num_labels = max(len(r["options"]) for r in train_records + val_records)
try:
    model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=num_labels)
except Exception:
    model = AutoModelForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=num_labels)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

best_loss = float("inf")
best_dir = Path("data/models/laya-techjob/best_checkpoint")
best_dir.mkdir(parents=True, exist_ok=True)

print("Starting training on GPU...")
global_step = 0
for epoch in range(3):
    model.train()
    for batch in train_loader:
        global_step += 1
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        if global_step % 50 == 0:
            model.eval()
            v_loss, v_corr, v_tot = 0.0, 0, 0
            with torch.no_grad():
                for vb in val_loader:
                    v_ids = vb["input_ids"].to(device)
                    v_msk = vb["attention_mask"].to(device)
                    v_lbl = vb["labels"].to(device)
                    with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                        vo = model(input_ids=v_ids, attention_mask=v_msk, labels=v_lbl)
                        v_loss += vo.loss.item() * len(v_lbl)
                    preds = vo.logits.argmax(dim=-1)
                    v_corr += (preds == v_lbl).sum().item()
                    v_tot += len(v_lbl)
            avg_vloss = v_loss / max(1, v_tot)
            acc = v_corr / max(1, v_tot)
            print(f"Epoch {epoch+1} | Step {global_step} | Val Loss: {avg_vloss:.4f} | Val Acc: {acc*100:.2f}%")
            if avg_vloss < best_loss:
                best_loss = avg_vloss
                model.save_pretrained(best_dir)
                tokenizer.save_pretrained(best_dir)
            model.train()

# Calibration
print("Calibrating temperature on holdout set...")
model = AutoModelForSequenceClassification.from_pretrained(best_dir).to(device)
model.eval()
all_logits, all_targets = [], []
with torch.no_grad():
    for vb in val_loader:
        vo = model(input_ids=vb["input_ids"].to(device), attention_mask=vb["attention_mask"].to(device))
        all_logits.extend(vo.logits.cpu().tolist())
        all_targets.extend(vb["labels"].cpu().tolist())

logits_arr = np.array(all_logits)
best_t, best_ece = 1.0, 1.0
for t in np.arange(0.1, 5.05, 0.05):
    s = logits_arr / t
    p = np.exp(s - np.max(s, axis=-1, keepdims=True))
    p = p / np.sum(p, axis=-1, keepdims=True)
    preds = np.argmax(p, axis=-1)
    confs = np.max(p, axis=-1)
    # 10-bin ECE
    bins = np.linspace(0, 1, 11)
    ece = 0.0
    for i in range(10):
        idx = np.where((confs >= bins[i]) & (confs <= bins[i+1]))[0]
        if len(idx) > 0:
            acc_bin = np.mean(preds[idx] == np.array(all_targets)[idx])
            conf_bin = np.mean(confs[idx])
            ece += (len(idx) / len(confs)) * abs(acc_bin - conf_bin)
    if ece < best_ece:
        best_ece = ece
        best_t = float(t)

print(f"Calibration finished! Optimal Temperature: {best_t:.2f} (ECE: {best_ece:.4f})")

out_final = Path("data/models/laya-techjob")
model.save_pretrained(out_final)
tokenizer.save_pretrained(out_final)
config = {
    "model_type": "laya-multilingual",
    "base_model": BASE_MODEL,
    "calibrated_temperature": best_t,
    "best_val_loss": best_loss,
    "calibrated_ece": best_ece
}
with open(out_final / "rl_agent_config.json", "w") as f:
    json.dump(config, f, indent=2)

with tarfile.open("data/models/laya-techjob.tar.gz", "w:gz") as tar:
    tar.add(out_final, arcname="laya-techjob")

print("Success! Bundled archive ready: data/models/laya-techjob.tar.gz")


## 4. Download Trained Bundle to Local Machine

In [ ]:
from google.colab import files
files.download("data/models/laya-techjob.tar.gz")
